# Data Preprocessing and Feature Engineering

This notebook implements a reproducible preprocessing pipeline for the suspicious review detection project.  
It ensures that all models operate on the same cleaned dataset and engineered features.

Key Objectives:
- Clean and standardise text data
- Engineer behavioural and linguistic features
- Ensure reproducibility across experiments
- Save processed datasets for reuse

A reproducible preprocessing pipeline was implemented to ensure consistency across all models. A single train-test split was generated and shared across Logistic Regression, SVM, and BERT based models to enable fair comparison of performance.

In [42]:
#Install necessary libraries
!pip install pandas numpy scikit-learn joblib

In [43]:
#Import necessary libraries
import pandas as pd
import numpy as np

# Machine learning 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Save files
import joblib
import os


## Load the dataset

In [44]:

# Load dataset
df = pd.read_csv("amazon_reviews_final.csv", low_memory= False)

# Inspect dataset
print(df.shape)
df.head()

(827398, 14)


,asin,category,reviewerID,reviewerName,overall,reviewText,unixReviewTime,reviewTime,verified,vote,reviewText_cleaned,reviewText_processed,actual_sentiment,predicted_sentiment
0,B00FLYWNYQ,Home & Kitchen,A0001624UKLQG4OFIM8X,Bob N,5.0,I was invited for a taste test of her 1st meal...,1501459200,"07 31, 2017",False,0,I was invited for a taste test of her 1st meal...,invite taste test meal pull pork turn unbeliev...,positive,positive
1,0316055433,Books,A0015414YF18MFZJ0N2T,Gail Martinez,5.0,"Terrific book, and a very long book! Reading ...",1398816000,"04 30, 2014",True,0,Terrific book and a very long book Reading it ...,terrific book long book read kindle didn t rea...,positive,positive
2,0316055433,Books,A0019380UVL8SJ13BYYX,Savannah,3.0,"Author had knowledge of New York City, antique...",1391817600,"02 8, 2014",False,0,Author had knowledge of New York City antiques...,author knowledge new york city antique people ...,negative,negative
3,0141353678,Books,A00336664SGESVX1FI20,Ejacobsen,5.0,It was a very good book. I could not put it do...,1400025600,"05 14, 2014",True,0,It was a very good book I could not put it dow...,good book not enjoy word romantic comical sad ...,positive,negative
4,0007420412,Books,A0037178T85I7MBSWLOR,Jaime Knapp,5.0,I ordered this book and it came pretty fast. I...,1370822400,"06 10, 2013",True,0,I ordered this book and it came pretty fast It...,order book come pretty fast exactly describe l...,positive,positive


# Feature Engineering

## Sentiment Features

These features capture the model’s predicted sentiment and identify inconsistencies between the predicted sentiment and actual rating, which may indicate potentially suspicious reviews.

In [45]:
# sentiment ratings mismatch
df["sentiment_rating_mismatch"] = (
    df["actual_sentiment"] != df["predicted_sentiment"]
).astype(int)

## Behavioural Features

These features capture user behaviour patterns such as unverified purchases and extreme ratings, which are commonly associated with potentially suspicious or biased reviews.

In [46]:
# Behavioural Feature

# Extreme ratings and unverified on the same review
df["extreme_unverified"] = (
    ((df["overall"] == 1) | (df["overall"] == 5)) & 
    (df["verified"] == False)
).astype(int)


#intermediate steps needed for ratios below
df["unverified_review"] = (df["verified"] == False).astype(int)
df["extreme_rating"] = ((df["overall"] == 1) | (df["overall"] == 5)).astype(int)

# Reviewer level aggregates (for training features)
df["extreme_rating_ratio"] = df.groupby("reviewerID")["extreme_rating"].transform("mean")
df["unverified_ratio"] = df.groupby("reviewerID")["unverified_review"].transform("mean")



## Time Based Features

These features capture temporal user behaviour, such as rapid successive reviews (burst activity) and overall review frequency, which may indicate automated or suspicious reviewing patterns.

In [47]:
# Time based feature

#Burst review
df["review_time"] = pd.to_datetime(df["unixReviewTime"], unit="s")

df = df.sort_values(["reviewerID", "review_time"]).reset_index(drop=True)

df["time_diff"] = df.groupby("reviewerID")["review_time"].diff().dt.total_seconds()

df["burst_review"] = (df["time_diff"] < 60).astype(int)

# Review frequency
df["review_frequency"] = df.groupby("reviewerID")["reviewerID"].transform("count")

## Linguistic Features

These features capture writing patterns such as short reviews, use of exclamation marks, pronoun usage, and generic language, which may indicate potentially deceptive review content.

In [48]:
#Linguistic features

# Very short reviews
# create word_count
df["word_count"] = df["reviewText_processed"].apply(lambda x: len(str(x).split()))

# create very_short_review
df["very_short_review"] = (df["word_count"] < 10).astype(int)

# exclamation marks
df["exclamation_count"] = df["reviewText"].astype(str).apply(lambda x: x.count("!"))

# personal pronouns
pronouns = ["i", "me", "my", "we", "us"]

df["personal_pronoun_count"] = df["reviewText_processed"].apply(
    lambda x: sum(word in str(x).split() for word in pronouns)
)

# generic phrases
generic_words = ["good", "great", "nice", "amazing"]

df["generic_word_flag"] = df["reviewText_processed"].apply(
    lambda x: int(any(word in str(x).split() for word in generic_words))
)& df["very_short_review"]

# average word length
df["avg_word_length"] = df["reviewText_processed"].apply(
    lambda x: np.mean([len(word) for word in str(x).split()]) if str(x).split() else 0
)

#lexical dieversity
# It measures how varied the vocabulary is in a review
# So a high score (close to 1) means the reviewer uses varied, descriptive language: more typical of a genuine review.
#A low score (close to 0) means the reviewer repeats the same words over and over: more typical of generic, bot, or AI-generated writing.

df["lexical_diversity"] = df["reviewText_processed"].apply(
    lambda x: len(set(str(x).split())) / max(len(str(x).split()), 1)
)

## AI Probability Score

This feature estimates the likelihood of a review being AI generated based on patterns such as overly structured text, excessive positive language, and lack of personal pronouns.

In [49]:
## AI probability score

def ai_probability_score(row):
    score = 0
    # use lowercase so matching is not case sensitive

    text = str(row["reviewText"]).lower()
    
    #very long reviews with no personalization
    if row["word_count"] > 100:
        score += 1

    # personal pronoun count  
    if row["personal_pronoun_count"] == 0:
        score += 1   

    # lexical diversity 
    if row["lexical_diversity"] < 0.4:
        score += 1  

    # excessive exclamation
    if row["exclamation_count"] >= 3: score += 1

    #overly polished  AI phrases       
    if any(word in text for word in ["i highly recommend", "five stars", "must buy", "excellent product"]):
        score += 1
     
        
    return score

df["ai_probability_score"] = df.apply(ai_probability_score, axis=1)

## Score based Labelling

In [50]:
# create score
df["suspicious_score"] = (
    df["sentiment_rating_mismatch"] +
    df["extreme_unverified"] +
    df["burst_review"]
)

#Set threshold
df["actual_suspicious_label"] = (df["suspicious_score"] >= 2).astype(int)

# check distribution
print(df["actual_suspicious_label"].value_counts(normalize=True))

actual_suspicious_label
0    0.951943
1    0.048057
Name: proportion, dtype: float64


In [51]:
df[["suspicious_score", "actual_suspicious_label"]].head()

,suspicious_score,actual_suspicious_label
0,1,0
1,0,0
2,0,0
3,1,0
4,0,0


In [52]:
#save to the dataset
final_df = df[[
    "asin",
    "category",
    "reviewerID",
    "reviewerName",
    "overall",
    "reviewText",
    "unixReviewTime",
    "reviewTime",
    "verified",
    "vote",
    "reviewText_cleaned",
    "reviewText_processed",
    "actual_sentiment",
    "predicted_sentiment",
    "actual_suspicious_label"
]]

# save clean dataset
final_df.to_csv("Amazon_review_final.csv", index=False)

## Features selection

In [53]:
features = [
    #reviewer behaviour
    "review_frequency",
    "extreme_rating_ratio",
    "unverified_ratio",

    # Linguistic quality
    "very_short_review",
    "lexical_diversity",
    "exclamation_count",
    "personal_pronoun_count",
    "generic_word_flag",

    #Ai likelihood
    "ai_probability_score"
]

X = df[features]
y = df["actual_suspicious_label"]

## Train/Test Split

In [54]:
# reset index
df = df.reset_index(drop=True)

# split using indices
train_index, test_index = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["actual_suspicious_label"]
)

print("Train size:", len(train_index))
print("Test size:", len(test_index))

Train size: 661918
Test size: 165480


In [55]:
# Structured features (FOR LOGISTIC & SVM)
X_train = df.loc[train_index, features]
X_test = df.loc[test_index, features]

# Labels
y_train = df.loc[train_index, "actual_suspicious_label"]
y_test = df.loc[test_index, "actual_suspicious_label"]


The dataset was split into training and testing sets using an 80/20 ratio.

- Training set (80%): 661,918 samples  
- Testing set (20%): 165,480 samples  

The split was performed using a fixed random_state=42 to ensure reproducibility, and stratify=y to maintain the same class distribution of suspicious and non suspicious reviews across both sets.

This approach ensures:
- Consistent results across runs  
- Fair model evaluation  
- Balanced representation of classes in both training and testing data  

## Text Preparation
TF-IDF (FOR LOGISTIC & SVM)

In [56]:

tfidf = TfidfVectorizer(max_features=5000)

X_train_text = tfidf.fit_transform(
    df.loc[train_index, "reviewText_processed"]
)

X_test_text = tfidf.transform(
    df.loc[test_index, "reviewText_processed"]
)

## Raw Text (For Bert)

In [57]:
# BERT uses raw text directly

train_text = df.loc[train_index, "reviewText"]
test_text = df.loc[test_index, "reviewText"]

# save shared data (reproducibility)

In [58]:
# create folder for suspicious pipeline
os.makedirs("shared_suspicious", exist_ok=True)

# Structured features
joblib.dump(X_train, "shared_suspicious/X_train_features.pkl")
joblib.dump(X_test, "shared_suspicious/X_test_features.pkl")

# TF-IDF features (for Logistic & SVM)
joblib.dump(X_train_text, "shared_suspicious/X_train_text.pkl")
joblib.dump(X_test_text, "shared_suspicious/X_test_text.pkl")

# Raw text (for BERT)
joblib.dump(train_text, "shared_suspicious/train_text.pkl")
joblib.dump(test_text, "shared_suspicious/test_text.pkl")

# Labels
joblib.dump(y_train, "shared_suspicious/y_train.pkl")
joblib.dump(y_test, "shared_suspicious/y_test.pkl")

# TF-IDF vectorizer
joblib.dump(tfidf, "shared_suspicious/tfidf_vectorizer.pkl")

#save indices
joblib.dump(train_index, "shared_suspicious/train_indices.pkl")
joblib.dump(test_index, "shared_suspicious/test_indices.pkl")

print("All suspicious pipeline files saved successfully ")

All suspicious pipeline files saved successfully 


## Final check

In [59]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nClass balance:")
print(y_train.value_counts(normalize=True))

Train shape: (661918, 9)
Test shape: (165480, 9)

Class balance:
actual_suspicious_label
0    0.951943
1    0.048057
Name: proportion, dtype: float64


## Class Distribution

The target variable shows a moderately imbalanced distribution:

- 0 (Not Suspicious): 95.2%  
- 1 (Suspicious): 4.8%  

This reflects a realistic scenario where most reviews are genuine, while a smaller portion are potentially suspicious. 

This imbalance will be addressed during model training using class_weight="balanced".


## Label Construction (Not Used as Training Features)

The ground truth label (`actual_suspicious_label`) is built from three independent signals that are excluded from model training:

1. **sentiment_rating_mismatch**:review text sentiment contradicts the star rating given
2. **extreme_unverified**: review has an extreme rating (1 or 5 stars) AND is unverified
3. **burst_review**: review was posted within 60 seconds of a previous review by the same user

A review is labelled suspicious (1) when its suspicious_score greater then or equals to 2, meaning at least two of these signals are present simultaneously.


## Training Features: Justification Table

| Feature | Category | What It Measures | Why It Detects Suspicious Reviews |
|---|---|---|---|
| predicted_sentiment_num | Sentiment | Converts predicted sentiment (positive/negative) to numeric (1/0) | Provides a machine readable signal of overall review tone for the model to learn from |
| review_frequency | Behavioural | Total number of reviews posted by the same reviewer | High-volume reviewers with no other signals may be automated bots or paid review farms (Jindal & Liu, 2008) |
| extreme_rating_ratio | Behavioural | Proportion of 1 star or 5 star ratings across all of a reviewer's reviews | Consistently extreme raters rarely reflect genuine purchasing behaviour, a high ratio is a strong signal of bias or manipulation (Ott et al., 2012) |
| unverified_ratio | Behavioural | Proportion of unverified purchases across all of a reviewer's reviews | Fake reviewers frequently have no verified purchase history, as they review products they never bought (He et al., 2011) |
| very_short_review | Linguistic | Flags reviews with fewer than 10 words | Genuine reviews tend to be descriptive, very short reviews often lack substance and are common in spam campaigns (Jindal & Liu, 2008) |
| lexical_diversity | Linguistic | Ratio of unique words to total words (vocabulary richness) | Low lexical diversity indicates repetitive or a writing tempelate, a  bot generated or AI generated review (Gehrmann et al., 2019) |
| exclamation_count | Linguistic | Number of exclamation marks in the review | Excessive enthusiasm through punctuation is a known indicator of deceptive or promotional writing (Ott et al., 2011) |
| personal_pronoun_count | Linguistic | Count of first person pronouns (I, me, my, we, us) | Genuine reviewers naturally reference their own experience, absence of personal pronouns suggests impersonal or AI generated text (Ott et al., 2011) |
| generic_word_flag | Linguistic | Flags presence of overused generic words (good, great, nice, amazing) | Fake reviews rely heavily on vague positive language rather than specific product details (Mukherjee et al., 2012) |
| ai_probability_score | AI Likelihood | Composite score (0-5) combining word count, pronouns, lexical diversity, exclamations, and AI phrases | Aggregates multiple weak signals into a single stronger indicator of AI generated or automated review content (Gehrmann et al., 2019) |


## References

### Academic Literature

Gehrmann, S., Strobelt, H., & Rush, A. M. (2019). GLTR: Statistical Detection and Visualization of Generated Text. *Proceedings of ACL 2019.* : Supports use of lexical diversity and linguistic patterns to identify AI generated text.

He, S., McAuley, J., & Leskovec, J. (2011). Ups and Downs: Modeling the Visual Evolution of Fashion Trends with One-Class Collaborative Filtering :Cited for unverified purchase patterns as indicators of fake reviews.

Jindal, N., & Liu, B. (2008). Opinion Spam and Analysis. *Proceedings of WSDM 2008.* : Foundational work on review spam detection; supports use of review frequency and short review flags.

Mukherjee, A., Liu, B., & Glance, N. (2012). Spotting Fake Reviewer Groups in Consumer Reviews. *Proceedings of WWW 2012.* : Supports use of generic language flags and behavioural patterns across reviewer histories.

Ott, M., Choi, Y., Cardie, C., & Hancock, J. T. (2011). Finding Deceptive Opinion Spam by Any Stretch of the Imagination. *Proceedings of ACL 2011.* : Supports use of personal pronouns, exclamation marks, and writing style features.

Ott, M., Cardie, C., & Hancock, J. T. (2012). Estimating the Prevalence of Deception in Online Review Communities. *Proceedings of WWW 2012.* :Supports extreme rating ratio as a behavioural indicator of suspicious activity.



### Official Documentation

McKinney, W. & PyData Development Team. (2024). *pandas documentation*. Retrieved from https://pandas.pydata.org/docs/ — Referenced for DataFrame operations, groupby transforms, and feature engineering implementation.

Harris, C. R., et al. (2020). Array programming with NumPy. *Nature, 585*, 357–362. Retrieved from https://numpy.org/doc/ — Referenced for numerical operations including `np.mean` and array handling in feature computation.

Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR, 12*, 2825–2830. Retrieved from https://scikit-learn.org/stable/documentation.html — Referenced for `train_test_split`, `TfidfVectorizer`, and preprocessing pipeline implementation.

Hugging Face. (2024). *bert-base-uncased model card*. Retrieved from https://huggingface.co/google-bert/bert-base-uncased — Referenced for BERT tokenisation and raw text preparation for transformer-based classification.


### Kaggle Notebooks

lazargugleta. (2022). *Amazon Review Spam Detection (ML/DL)*. Kaggle. Retrieved from https://www.kaggle.com/code/lazargugleta/amazon-review-spam-detection-ml-dl :Referenced for feature engineering and ML/DL pipeline structure applied to Amazon review spam detection.

### AI Tools

OpenAI. (2025). *ChatGPT (GPT-4o)*. OpenAI. Retrieved from https://chatgpt.com — Used to support debugging of preprocessing code and to clarify technical concepts including feature leakage, lexical diversity computation, and TF-IDF vectorisation.
